# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library and the [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Print basic metadata
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets, field ids, and columns
record_sets = list(dataset.record_sets)

for rs in record_sets:
    rs_id = rs['@id']
    print(f"\nRecord Set '@id': {rs_id}")
    name = rs.get('name', 'N/A')
    print(f"  Name: {name}")
    fields = rs.get('field', [])
    if isinstance(fields, dict): fields = [fields]
    print("  Fields:")
    for f in fields:
        fid = f.get('@id') if isinstance(f, dict) else f
        print(f"   - {fid}")
    columns = rs.get('column', [])
    if isinstance(columns, dict): columns = [columns]
    if columns:
        print("  Columns (@id):")
        for c in columns:
            cid = c.get('@id') if isinstance(c, dict) else c
            print(f"    - {cid}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare dictionary of DataFrames by record set '@id'
record_sets = list(dataset.record_sets)
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    try:
        df = pd.DataFrame(dataset.records(record_set=rs_id))
        if not df.empty:
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for Record Set '@id': {rs_id}")
            print(df.head(2))
    except Exception as e:
        print(f"[Warning] Could not load records for {rs_id}: {e}")

# Pick the primary record set for analysis (the one with most records/columns)
max_rs_id = None
max_cols = 0
for rs_id, df in dataframes.items():
    if df.shape[1] > max_cols:
        max_cols = df.shape[1]
        max_rs_id = rs_id

if max_rs_id:
    print(f"\nUsing record set '@id': {max_rs_id} with columns: \n{dataframes[max_rs_id].columns.tolist()}")
    display(dataframes[max_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA - adjust field IDs based on overview output
# Pick a numeric field and a group field based on the loaded DataFrame
df = dataframes[max_rs_id] if max_rs_id in dataframes else None

if df is not None and not df.empty:
    # Try to auto-detect a numeric field:
    numeric_fields = [c for c in df.columns 
                     if pd.api.types.is_numeric_dtype(df[c]) and not c.startswith('@')]
    if not numeric_fields:
        # If no numeric type detected (e.g., all string), try casting common names
        maybe_numeric = [c for c in df.columns if 'count' in c.lower() or 'abundance' in c.lower() or 'mw' in c.lower() or 'coverage' in c.lower()]
        for col in maybe_numeric:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except:
                pass
        numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    # Pick the first numeric field for the example
    numeric_field = numeric_fields[0] if numeric_fields else df.columns[0]
    print(f"Using numeric field: {numeric_field}\n")
    # Filter: choose a threshold (for demonstration, the mean)
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (total: {len(filtered_df)})")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Example groupby: try to pick a categorical field
    group_field = None
    cat_candidates = [c for c in df.columns if (df[c].dtype == 'object' and c != numeric_field)]
    if cat_candidates:
        group_field = cat_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group_field exists and is not too high cardinality, boxplot
    if group_field and df[group_field].nunique() < 30:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df.dropna(subset=[group_field, numeric_field]))
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we used `mlcroissant` to load and explore the FAIR² mass spectrometry dataset via its Croissant schema. We:
- Discovered available record sets and their fields by `@id`.
- Loaded records into pandas DataFrames.
- Performed basic filtering, normalization, and grouping operations on numeric fields.
- Visualized data distributions and relationships between features.

For further analysis, additional domain-specific fields and methods could be used to filter and model protein abundance, modifications, or other biological properties.